In [1]:
from pathlib import Path
import pandas as pd
import requests

# Locate the project root
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

# Create a folder for raw demographic data
demographic_raw_dir = (
    project_root
    / "data"
    / "raw"
    / "demographics"
)

demographic_raw_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:")
print(project_root)

print("\nDemographic raw-data folder:")
print(demographic_raw_dir)

Project root:
c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection

Demographic raw-data folder:
c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\raw\demographics


In [5]:
from getpass import getpass

# 2020–2024 ACS 5-Year Estimates
census_url = (
    "https://api.census.gov/data/"
    "2024/acs/acs5"
)

# Variables requested from the Census API
census_variables = [
    "NAME",

    # Total population
    "B01003_001E",
    "B01003_001M",

    # Asian alone population
    "B02001_005E",
    "B02001_005M",

    # Chinese, except Taiwanese
    "B02015_002E",
    "B02015_002M",

    # Taiwanese
    "B02015_008E",
    "B02015_008M",

    # Median household income
    "B19013_001E",
    "B19013_001M"
]

# Los Angeles County Census Tracts
census_parameters = {
    "get": ",".join(census_variables),
    "for": "tract:*",
    "in": "state:06 county:037"
}

# Enter the API key privately
census_api_key = getpass(
    "Paste your Census API key, then press Enter: "
).strip()

census_parameters["key"] = census_api_key

# Send the request
response = requests.get(
    census_url,
    params=census_parameters,
    timeout=60
)

response.raise_for_status()

# Confirm that the response is real JSON data
response_type = response.headers.get(
    "content-type",
    ""
)

print("Request status:", response.status_code)
print("Response type:", response_type)

if "application/json" not in response_type:
    raise RuntimeError(
        "The Census API did not return JSON data. "
        "Please check the API key."
    )

Request status: 200
Response type: application/json;charset=utf-8


In [6]:
import json

# Convert the API response to Python data
census_json = response.json()

# Save an untouched copy of the API response
raw_json_file = (
    demographic_raw_dir
    / "acs_2024_la_county_tracts_raw.json"
)

with open(
    raw_json_file,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        census_json,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Raw JSON saved to:")
print(raw_json_file)

print(
    "Census tract rows returned:",
    len(census_json) - 1
)

Raw JSON saved to:
c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\raw\demographics\acs_2024_la_county_tracts_raw.json
Census tract rows returned: 2498


In [7]:
# Convert the Census JSON response into a DataFrame

acs = pd.DataFrame(
    census_json[1:],
    columns=census_json[0]
)

# Rename Census variable codes to understandable names
acs = acs.rename(
    columns={
        "NAME": "tract_name",

        "B01003_001E": "total_population",
        "B01003_001M": "total_population_moe",

        "B02001_005E": "asian_alone",
        "B02001_005M": "asian_alone_moe",

        "B02015_002E": "chinese_except_taiwanese",
        "B02015_002M": "chinese_except_taiwanese_moe",

        "B02015_008E": "taiwanese_alone",
        "B02015_008M": "taiwanese_alone_moe",

        "B19013_001E": "median_household_income",
        "B19013_001M": "median_household_income_moe"
    }
)

# Create the 11-digit Census Tract identifier
acs["geoid"] = (
    acs["state"]
    + acs["county"]
    + acs["tract"]
)

print("Rows:", len(acs))
print("Columns:", len(acs.columns))
print("Unique GEOIDs:", acs["geoid"].nunique())

display(acs.head())

Rows: 2498
Columns: 15
Unique GEOIDs: 2498


,tract_name,total_population,total_population_moe,asian_alone,asian_alone_moe,chinese_except_taiwanese,chinese_except_taiwanese_moe,taiwanese_alone,taiwanese_alone_moe,median_household_income,median_household_income_moe,state,county,tract,geoid
0,Census Tract 1011.10; Los Angeles County; Cali...,4294,436,336,168,25,23,18,29,88438,20688,06,037,101110,06037101110
1,Census Tract 1011.22; Los Angeles County; Cali...,4124,876,509,179,49,47,0,14,104643,40990,06,037,101122,06037101122
2,Census Tract 1012.20; Los Angeles County; Cali...,3467,423,484,224,0,14,0,14,92317,21248,06,037,101220,06037101220
3,Census Tract 1012.21; Los Angeles County; Cali...,3593,599,185,158,0,14,0,14,52204,17943,06,037,101221,06037101221
4,Census Tract 1012.22; Los Angeles County; Cali...,2290,438,46,50,10,15,0,14,38125,37622,06,037,101222,06037101222


In [12]:
# Convert estimate and margin-of-error columns to numeric values

numeric_columns = [
    "total_population",
    "total_population_moe",
    "asian_alone",
    "asian_alone_moe",
    "chinese_except_taiwanese",
    "chinese_except_taiwanese_moe",
    "taiwanese_alone",
    "taiwanese_alone_moe",
    "median_household_income",
    "median_household_income_moe"
]

acs[numeric_columns] = acs[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

print("Duplicate GEOIDs:", acs["geoid"].duplicated().sum())
print("Missing GEOIDs:", acs["geoid"].isna().sum())
print("Zero-population tracts:", (acs["total_population"] == 0).sum())

negative_value_summary = (
    (acs[numeric_columns] < 0)
    .sum()
    .rename_axis("column")
    .reset_index(name="negative_value_count")
)

display(
    negative_value_summary.loc[
        negative_value_summary["negative_value_count"] > 0
    ]
)

Duplicate GEOIDs: 0
Missing GEOIDs: 0
Zero-population tracts: 21


,column,negative_value_count
8,median_household_income,42
9,median_household_income_moe,58


In [13]:
# Replace Census special negative income codes with missing values

income_columns = [
    "median_household_income",
    "median_household_income_moe"
]

acs[income_columns] = acs[income_columns].mask(
    acs[income_columns] < 0
)

print(
    "Missing median household income:",
    acs["median_household_income"].isna().sum()
)

print(
    "Missing median household income MOE:",
    acs["median_household_income_moe"].isna().sum()
)

print(
    "Remaining negative values:",
    (acs[income_columns] < 0).sum().sum()
)

Missing median household income: 42
Missing median household income MOE: 58
Remaining negative values: 0


In [14]:
import numpy as np

# Combine Chinese-except-Taiwanese and Taiwanese estimates

acs["chinese_total_estimate"] = (
    acs["chinese_except_taiwanese"]
    + acs["taiwanese_alone"]
)

# Approximate the combined margin of error
acs["chinese_total_moe"] = np.sqrt(
    acs["chinese_except_taiwanese_moe"] ** 2
    + acs["taiwanese_alone_moe"] ** 2
)

# Prevent division by zero in zero-population tracts
population_denominator = acs["total_population"].replace(0, np.nan)
asian_denominator = acs["asian_alone"].replace(0, np.nan)

# Calculate demographic percentages
acs["asian_share_pct"] = (
    acs["asian_alone"]
    / population_denominator
    * 100
)

acs["chinese_share_pct"] = (
    acs["chinese_total_estimate"]
    / population_denominator
    * 100
)

acs["chinese_share_of_asian_pct"] = (
    acs["chinese_total_estimate"]
    / asian_denominator
    * 100
)

# Round percentages for easier inspection
percentage_columns = [
    "asian_share_pct",
    "chinese_share_pct",
    "chinese_share_of_asian_pct"
]

acs[percentage_columns] = acs[percentage_columns].round(2)

display(
    acs[
        [
            "geoid",
            "tract_name",
            "total_population",
            "asian_alone",
            "chinese_total_estimate",
            "chinese_total_moe",
            "asian_share_pct",
            "chinese_share_pct",
            "chinese_share_of_asian_pct",
            "median_household_income"
        ]
    ].head()
)

,geoid,tract_name,total_population,asian_alone,chinese_total_estimate,chinese_total_moe,asian_share_pct,chinese_share_pct,chinese_share_of_asian_pct,median_household_income
0,06037101110,Census Tract 1011.10; Los Angeles County; Cali...,4294,336,43,37.013511,7.82,1.00,12.80,88438.0
1,06037101122,Census Tract 1011.22; Los Angeles County; Cali...,4124,509,49,49.040799,12.34,1.19,9.63,104643.0
2,06037101220,Census Tract 1012.20; Los Angeles County; Cali...,3467,484,0,19.798990,13.96,0.00,0.00,92317.0
3,06037101221,Census Tract 1012.21; Los Angeles County; Cali...,3593,185,0,19.798990,5.15,0.00,0.00,52204.0
4,06037101222,Census Tract 1012.22; Los Angeles County; Cali...,2290,46,10,20.518285,2.01,0.44,21.74,38125.0


In [15]:
# Quality checks for the derived demographic indicators

print("Total Census tracts:", len(acs))

print(
    "Chinese estimate greater than Asian estimate:",
    (acs["chinese_total_estimate"] > acs["asian_alone"]).sum()
)

print(
    "Chinese estimate greater than total population:",
    (acs["chinese_total_estimate"] > acs["total_population"]).sum()
)

print(
    "Chinese share greater than 100%:",
    (acs["chinese_share_pct"] > 100).sum()
)

print(
    "Missing Chinese share:",
    acs["chinese_share_pct"].isna().sum()
)

print(
    "Chinese MOE greater than estimate:",
    (
        acs["chinese_total_moe"]
        > acs["chinese_total_estimate"]
    ).sum()
)

print(
    "Maximum Chinese share:",
    acs["chinese_share_pct"].max()
)

Total Census tracts: 2498
Chinese estimate greater than Asian estimate: 0
Chinese estimate greater than total population: 0
Chinese share greater than 100%: 0
Missing Chinese share: 21
Chinese MOE greater than estimate: 1529
Maximum Chinese share: 78.94


In [16]:
# Measure the reliability of the combined Chinese population estimate

acs["chinese_relative_moe"] = (
    acs["chinese_total_moe"]
    / acs["chinese_total_estimate"].replace(0, np.nan)
)

# Create an approximate 90% confidence interval
acs["chinese_estimate_lower"] = (
    acs["chinese_total_estimate"]
    - acs["chinese_total_moe"]
).clip(lower=0)

acs["chinese_estimate_upper"] = (
    acs["chinese_total_estimate"]
    + acs["chinese_total_moe"]
)

# Assign a reliability category
acs["chinese_estimate_reliability"] = "high_uncertainty"

acs.loc[
    acs["chinese_relative_moe"] <= 1,
    "chinese_estimate_reliability"
] = "usable_with_caution"

acs.loc[
    acs["chinese_relative_moe"] <= 0.5,
    "chinese_estimate_reliability"
] = "more_reliable"

acs.loc[
    acs["chinese_total_estimate"] == 0,
    "chinese_estimate_reliability"
] = "zero_estimate"

acs.loc[
    acs["total_population"] == 0,
    "chinese_estimate_reliability"
] = "not_applicable"

display(
    acs["chinese_estimate_reliability"]
    .value_counts()
    .rename_axis("reliability")
    .reset_index(name="tract_count")
)

,reliability,tract_count
0,high_uncertainty,905
1,usable_with_caution,669
2,zero_estimate,603
3,more_reliable,300
4,not_applicable,21


In [17]:
# Compare Chinese population concentration and market size

comparison_columns = [
    "geoid",
    "tract_name",
    "total_population",
    "chinese_total_estimate",
    "chinese_total_moe",
    "chinese_share_pct",
    "median_household_income",
    "chinese_estimate_reliability"
]

top_by_share = (
    acs.loc[
        acs["chinese_estimate_reliability"].isin(
            ["more_reliable", "usable_with_caution"]
        ),
        comparison_columns
    ]
    .sort_values(
        ["chinese_share_pct", "chinese_total_estimate"],
        ascending=[False, False]
    )
    .head(15)
)

top_by_population = (
    acs.loc[
        acs["chinese_estimate_reliability"].isin(
            ["more_reliable", "usable_with_caution"]
        ),
        comparison_columns
    ]
    .sort_values(
        ["chinese_total_estimate", "chinese_share_pct"],
        ascending=[False, False]
    )
    .head(15)
)

print("Top tracts by Chinese population share:")
display(top_by_share)

print("Top tracts by estimated Chinese population:")
display(top_by_population)

Top tracts by Chinese population share:


,geoid,tract_name,total_population,chinese_total_estimate,chinese_total_moe,chinese_share_pct,median_household_income,chinese_estimate_reliability
1544,06037481711,Census Tract 4817.11; Los Angeles County; Cali...,4388,3464,702.728966,78.94,51655.0,more_reliable
1257,06037403407,Census Tract 4034.07; Los Angeles County; Cali...,2368,1661,343.739727,70.14,156552.0,more_reliable
1545,06037481712,Census Tract 4817.12; Los Angeles County; Cali...,5269,3580,908.198767,67.94,46387.0,more_reliable
1555,06037482201,Census Tract 4822.01; Los Angeles County; Cali...,3813,2468,513.515336,64.73,62223.0,more_reliable
1546,06037481713,Census Tract 4817.13; Los Angeles County; Cali...,2892,1846,465.966737,63.83,74453.0,more_reliable
1502,06037464101,Census Tract 4641.01; Los Angeles County; Cali...,2223,1376,291.650476,61.90,250001.0,more_reliable
1547,06037481714,Census Tract 4817.14; Los Angeles County; Cali...,2595,1606,251.872984,61.89,42675.0,more_reliable
1540,06037481603,Census Tract 4816.03; Los Angeles County; Cali...,3711,2179,486.201604,58.72,93375.0,more_reliable
1410,06037431600,Census Tract 4316; Los Angeles County; California,3941,2307,529.419493,58.54,120511.0,more_reliable
1249,06037403325,Census Tract 4033.25; Los Angeles County; Cali...,4706,2735,531.294645,58.12,111445.0,more_reliable


Top tracts by estimated Chinese population:


,geoid,tract_name,total_population,chinese_total_estimate,chinese_total_moe,chinese_share_pct,median_household_income,chinese_estimate_reliability
1545,06037481712,Census Tract 4817.12; Los Angeles County; Cali...,5269,3580,908.198767,67.94,46387.0,more_reliable
1372,06037408707,Census Tract 4087.07; Los Angeles County; Cali...,6839,3532,1016.854955,51.64,134400.0,more_reliable
1544,06037481711,Census Tract 4817.11; Los Angeles County; Cali...,4388,3464,702.728966,78.94,51655.0,more_reliable
1396,06037430801,Census Tract 4308.01; Los Angeles County; Cali...,7166,3315,851.597323,46.26,96042.0,more_reliable
1411,06037431701,Census Tract 4317.01; Los Angeles County; Cali...,6188,3081,874.739390,49.79,117887.0,more_reliable
1567,06037482600,Census Tract 4826; Los Angeles County; California,6611,3062,1067.169152,46.32,114493.0,more_reliable
1248,06037403324,Census Tract 4033.24; Los Angeles County; Cali...,6418,2944,742.830398,45.87,114547.0,more_reliable
1417,06037432102,Census Tract 4321.02; Los Angeles County; Cali...,5793,2890,605.528695,49.89,102500.0,more_reliable
1367,06037408628,Census Tract 4086.28; Los Angeles County; Cali...,5463,2858,568.546392,52.32,88021.0,more_reliable
1504,06037464200,Census Tract 4642; Los Angeles County; California,5451,2846,562.538888,52.21,190991.0,more_reliable


In [21]:
from pathlib import Path

# Add income quality indicators

acs["income_top_coded"] = (
    acs["median_household_income"] >= 250001
)

acs["income_relative_moe"] = (
    acs["median_household_income_moe"]
    / acs["median_household_income"]
)

# Keep GEOID components as text so leading zeros are preserved
acs["state"] = acs["state"].astype("string")
acs["county"] = acs["county"].astype("string")
acs["tract"] = acs["tract"].astype("string")
acs["geoid"] = acs["geoid"].astype("string")

# Arrange the columns for the cleaned output
final_columns = [
    "geoid",
    "state",
    "county",
    "tract",
    "tract_name",
    "total_population",
    "total_population_moe",
    "asian_alone",
    "asian_alone_moe",
    "asian_share_pct",
    "chinese_except_taiwanese",
    "chinese_except_taiwanese_moe",
    "taiwanese_alone",
    "taiwanese_alone_moe",
    "chinese_total_estimate",
    "chinese_total_moe",
    "chinese_estimate_lower",
    "chinese_estimate_upper",
    "chinese_relative_moe",
    "chinese_estimate_reliability",
    "chinese_share_pct",
    "chinese_share_of_asian_pct",
    "median_household_income",
    "median_household_income_moe",
    "income_relative_moe",
    "income_top_coded"
]

acs_clean = acs[final_columns].copy()

# Create the output folder and save the processed tract data
output_directory = Path("data/processed")
output_directory.mkdir(parents=True, exist_ok=True)

output_path = (
    output_directory
    / "acs_2024_la_county_tract_demographics.csv"
)

acs_clean.to_csv(
    output_path,
    index=False
)

print("Rows saved:", len(acs_clean))
print("Unique GEOIDs:", acs_clean["geoid"].nunique())
print("File saved to:", output_path.resolve())

Rows saved: 2498
Unique GEOIDs: 2498
File saved to: C:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\notebooks\data\processed\acs_2024_la_county_tract_demographics.csv
